# derived_8.4-eval-1.3 — Leave-One-Station-Out (LOSO) Spatial Generalization with 54-Backbone Two-Regime Clustering

This experiment is `derived_8.4-eval-1.2` (2 baselines + 5 MoE routing strategies × 9 per-regime delta-grid points = 47 configurations, all sharing the same 54-feature backbone / V0-50 baseline / candidate pool / XGBoost hyperparameters) **plus one new strategy**: `Clustering_Backbone54_k2` — KMeans(k=2) fitted on the **same 54 shared-backbone features as the single-regime global model** (`Global_Single_54`), with the same 9-point delta grid (56 configurations total). The two-regime model is therefore a direct development of the single-regime model (same 54 features for routing *and* base features); no separate V0-50 feature source needs to be explained. Its 9 grid points reuse eval-1.1's `Clustering_V0_Full_k2` per-(c0, c1) delta additions, and its winner is pinned to the same grid point that won for V0-Full in eval-1.1 (c0=0, c1=10), decided before running.

Execution follows the `derived_8.4-eval-2.0` **parallel worker format**: the driver spawns 8 `run_loso_worker.py` subprocesses, each training one (config, station) fold on the H100 GPU concurrently (scheduled via `sbatch run_slurm.sh`). For each of the 56 configurations and each held-out station $s$: the router is refitted and experts are trained on the trainval rows of the **remaining 6 stations** (train 2017–2020 + val 2021–2022), then evaluated on all test rows of station $s$ (2023–2025). Per-configuration × per-station metrics are collected, so station difficulty is a direct byproduct.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
from IPython.display import Image, display

# Robust resolution of the experiment dir whether executed from notebooks/ or in-place.
candidates = [Path.cwd() / "experiment/derived_8.4-eval-1.3", Path.cwd()]
EXP_DIR = next((p for p in candidates if (p / "loso_config_summary.csv").exists()), Path.cwd())

df_summary = pd.read_csv(EXP_DIR / "loso_config_summary.csv")
df_station = pd.read_csv(EXP_DIR / "loso_station_summary.csv")
df_pcs = pd.read_csv(EXP_DIR / "loso_per_config_station.csv")
df_regime = pd.read_csv(EXP_DIR / "loso_per_regime_metrics.csv")
df_year = pd.read_csv(EXP_DIR / "loso_per_year_metrics.csv")

print(f"Loaded {len(df_summary)} configurations, {len(df_station)} stations, "
      f"{len(df_pcs)} config×station folds, {len(df_regime)} regime rows, {len(df_year)} year rows.")


## LOSO Configuration Leaderboard

Mean $R^2$ across the 7 held-out stations (each model trained on the other 6 stations, tested on the held-out one). `loso_mean_r2` is the average of per-station $R^2$; `loso_pooled_r2` is the sample-count-weighted $R^2$ over the concatenated 6,620 held-out test samples (directly comparable to eval-1.1's pooled test $R^2$); `loso_min/max_r2` show the spread across stations. `temporal_test_r2` is the configuration's *temporal* test $R^2$ — from `derived_8.4-eval-1.1` for the 47 pinned configurations, and from the full-training baseline (`full_config_summary.csv`) for the 9 new `Clustering_Backbone54_k2` configurations (no eval-1.1 row exists); `loso_minus_test_r2` is the spatial-generalization gap (mean-of-stations version).


In [ ]:
leaderboard_cols = [
    "config_label", "strategy_name", "loso_mean_r2", "loso_pooled_r2", "loso_std_r2",
    "loso_min_r2", "loso_max_r2", "loso_mean_rmse", "loso_mean_bias", "temporal_test_r2",
    "loso_minus_test_r2", "is_winner",
]
print("### LOSO Configuration Leaderboard (mean R² over 7 held-out stations)")
print(df_summary[leaderboard_cols].to_markdown(index=False))


## Station Difficulty (Byproduct)

Aggregating LOSO $R^2$ over all 56 configurations per held-out station reveals which stations are intrinsically harder to generalize to (fewer stations share their climate/soil regime). `n_negative_r2` counts how many of the 56 configurations produced a negative $R^2$ when that station was held out.


In [ ]:
station_cols = ["station", "n_configs", "total_test_n", "median_r2", "mean_r2", "std_r2",
                "min_r2", "max_r2", "mean_rmse", "mean_bias", "n_negative_r2"]
print("### Station Difficulty — LOSO R² aggregated over all configurations")
print(df_station[station_cols].to_markdown(index=False))


## Per-Configuration × Per-Station R² Matrix

Full LOSO $R^2$ matrix (rows = the 56 configurations, columns = held-out stations). This is the raw per-configuration metric collection that makes station difficulty directly attributable per model.


In [ ]:
piv = df_pcs.pivot_table(index="config_label", columns="station", values="r2")
station_order = df_station.sort_values("median_r2", ascending=False)["station"].tolist()
piv = piv[station_order]
print("### LOSO R² — Configuration × Held-out Station")
print(piv.to_markdown())


## Per-Regime (Cluster) Breakdown under LOSO

Cluster-level metrics on the held-out stations for the **winning** configurations only (the 5 eval-1.1 winners + the pinned `Clustering_Backbone54_k2` winner c0=0, c1=10). Regimes are the $K=2$ partitions produced by each strategy's router, refitted on the 6 training stations per fold.


In [ ]:
winners = df_summary.loc[df_summary["is_winner"], "config_id"].tolist()
df_w = df_regime[df_regime["config_id"].isin(winners)].copy()
df_w = df_w.merge(df_summary[["config_id", "config_label"]], on="config_id", how="left")
regime_cols = ["config_label", "station", "cluster", "n_train", "n_test", "r2", "rmse", "ubrmse", "bias", "mae"]
print("### Per-Regime LOSO Metrics (winning configurations)")
print(df_w[regime_cols].to_markdown(index=False))


## Yearly Breakdown under LOSO

For each configuration, mean held-out-station $R^2$ per test year (2023–2025). Aggregated over the 7 stations, this shows how spatial generalization evolves over time.


In [ ]:
df_y = df_year.groupby(["config_id", "year"])["r2"].mean().reset_index()
df_y = df_y.merge(df_summary[["config_id", "config_label", "strategy_name"]], on="config_id", how="left")
year_piv = df_y.pivot_table(index="config_label", columns="year", values="r2")
year_piv = year_piv.reindex(df_summary["config_label"].tolist())
print("### Yearly Mean LOSO R² by Configuration")
print(year_piv.to_markdown())


## Figures

Four figures summarize the LOSO results: (1) configuration × station $R^2$ heatmap, (2) LOSO-mean $R^2$ bar chart per configuration with per-station min/max whiskers, (3) station difficulty bar chart, and (4) boxplot of per-configuration $R^2$ per held-out station.


In [ ]:
for fig in [
    "loso_r2_config_station_heatmap.png",
    "loso_r2_config_summary.png",
    "loso_station_difficulty.png",
    "loso_r2_station_boxplot.png",
]:
    display(Image(filename=str(EXP_DIR / fig)))


## Single-Regime → Two-Regime Development (`Clustering_Backbone54_k2`)

The two-regime model is presented as a **development of the single-regime model**, not a separate architecture: the router (KMeans k=2) and both specialists use the **same 54 shared-backbone features** as `Global_Single_54`, with only the per-cluster delta additions (pinned from eval-1.1's `Clustering_V0_Full_k2` winner) added to the second specialist. This avoids having to explain a separate V0-50 feature source. The development line below compares the single-regime global model, the new two-regime model (pinned winner c0=0, c1=10), and the eval-1.1 `Clustering_V0_Full_k2` winner (the previous best spatial generalizer), followed by the new strategy's full 9-point delta grid.


In [ ]:
dev_ids = [
    "Global_Single_54",
    "Clustering_Backbone54_k2_c0_0_c1_10",
    "Clustering_V0_Full_k2_c0_0_c1_10",
]
dev = df_summary[df_summary["config_id"].isin(dev_ids)].copy()
dev["line"] = ["single-regime global (54 feats)", "two-regime 54-backbone (NEW, winner)", "two-regime V0-routed (eval-1.1 winner)"]
dev_cols = ["line", "loso_mean_r2", "loso_pooled_r2", "loso_min_r2", "loso_max_r2",
            "loso_mean_rmse", "loso_mean_bias", "temporal_test_r2", "loso_minus_test_r2"]
print("### Development line: single-regime 54 → two-regime 54")
print(dev[dev_cols].to_markdown(index=False))

grid54 = df_summary[df_summary["strategy_name"] == "Clustering_Backbone54_k2"].sort_values(
    ["cluster_0_count", "cluster_1_count"])
grid_cols = ["config_label", "loso_mean_r2", "loso_pooled_r2", "loso_mean_rmse",
             "loso_mean_bias", "temporal_test_r2", "is_winner"]
print("\n### Clustering_Backbone54_k2 — full 9-point delta grid")
print(grid54[grid_cols].to_markdown(index=False))


## Station Similarity & Clustering — Spatial-Generalization Hypothesis

**Hypothesis.** A station generalizes well spatially when the LOSO training set contains another station with *similar climate and geography* ("a twin"), and poorly when the station *stands out* from the rest. This section builds per-station feature vectors — WorldClim BioClim BIO1-19, geography, and soil texture from `data/splits/derived_8.4/station_static_features.csv`, plus observed climatology (mean/std of precip, LST, NDVI, SMAP, soil moisture over train+val) — then clusters the 7 stations (Ward + PCA) and tests the hypothesis by correlating per-station isolation metrics against two LOSO difficulty measures: the median R² over all 56 configurations and the per-station R² of the best spatial generalizer (the pinned `Clustering_Backbone54_k2` winner). Implementation lives in `eval13/station_sim.py`; every number below is computed by this notebook.


In [ ]:
import sys
import yaml

sys.path.insert(0, str(EXP_DIR.resolve()))
from eval13 import station_sim as ss

with open(EXP_DIR / "config.yaml") as f:
    _cfg = yaml.safe_load(f)
PROJECT_ROOT = Path("../../..").resolve()

feats = ss.build_station_features(PROJECT_ROOT, _cfg)
print(f"Feature sets: {len(ss.GEO_FEATURES)} geography, {len(ss.BIOCLIM_FEATURES)} BioClim, "
      f"{len(ss.SOIL_FEATURES)} soil, {len(ss.DYNAMIC_FEATURES)} observed-climatology "
      f"({len(ss.STATIC_FEATURES) + len(ss.DYNAMIC_FEATURES)} total).")
print(f"Land cover: {dict(feats.landcover)}")

diff = ss.load_difficulty(EXP_DIR)
profile = ss.station_profile_table(feats, {"median_r2": diff["median_r2"]})
print("\n### Station profiles (MAT = annual mean temp °C×10, MAP = annual precip mm)")
print(profile.to_markdown())


### Pairwise station distance

Euclidean distance on the z-scored 39-feature vector (29 static + 10 observed climatology). `nn_dist` (1-NN) is the distance to the closest *other* station — a small value means the station has a climate/geography "twin" in the dataset.


In [ ]:
X_std = ss.standardize(feats.combined)
dist = ss.pairwise_distance(X_std)
print("### Pairwise Euclidean distance (standardized static + observed features)")
print(dist.round(2).to_markdown())


### Hierarchical clustering and PCA

Ward linkage on the same standardized features; flat cluster assignments at k = 2 and k = 3 (k = 2 mirrors the MoE routers' two-regime split). PCA gives a 2-D view of the station embedding.


In [ ]:
Z = ss.ward_linkage(X_std)
order = ss.leaf_order(Z, list(X_std.index))
print("Dendrogram leaf order:", order)
for k in (2, 3):
    print(f"\n### Ward clusters (k={k})")
    cl = ss.cluster_labels(Z, k, list(X_std.index)).to_frame("cluster")
    cl["median_r2"] = diff["median_r2"]
    print(cl.to_markdown())
scores, ev, loadings = ss.pca_projection(X_std)
print(f"\nPCA explained variance: PC1 = {ev[0]:.1%}, PC2 = {ev[1]:.1%} (total {ev.sum():.1%})")


In [ ]:
iso = ss.isolation_scores(X_std)
tbl = iso.join(diff[["median_r2", "mean_r2", "winner_r2"]])
tbl = tbl.sort_values("median_r2", ascending=False)
print(f"Winner config used for `winner_r2`: {diff['winner_config'].iloc[0]}")
print("\n### Isolation metrics vs LOSO difficulty")
print(tbl.round(3).to_markdown())
print("\n### Spearman rank correlation (isolation × difficulty)")
print(ss.spearman_table(iso, diff).to_markdown(index=False))


In [ ]:
# Sensitivity: static features only (geography + BioClim + soil; no target-derived
# observed climatology) — confirms the correlation pattern is not an artifact of
# including mean/std soil moisture in the feature vector.
X_static = ss.standardize(feats.static)
iso_static = ss.isolation_scores(X_static)
print("### Sensitivity — static features only (geography + BioClim + soil)")
print(ss.spearman_table(iso_static, diff).to_markdown(index=False))


### Figures

Five figures summarize the station similarity structure and the hypothesis test: (1) pairwise similarity heatmap in dendrogram order, (2) Ward dendrogram with leaf colors by difficulty tier, (3) PCA embedding colored by median LOSO R² with the top feature loadings, (4) isolation-vs-difficulty scatter (1-NN / mean distance × median / winner-config R², Spearman ρ annotated), and (5) the standardized feature matrix showing which features make the hard stations extreme.


In [ ]:
sim = ss.similarity_matrix(dist)
for fig_path in [
    ss.plot_similarity_heatmap(sim, order, EXP_DIR),
    ss.plot_dendrogram(Z, list(X_std.index), diff, EXP_DIR),
    ss.plot_pca_scatter(scores, ev, loadings, diff, EXP_DIR),
    ss.plot_isolation_vs_difficulty(iso, diff, EXP_DIR),
    ss.plot_feature_heatmap(X_std, order, EXP_DIR),
]:
    display(Image(filename=str(fig_path)))
print("Saved 5 station-similarity figures to the experiment directory.")


### Interpretation

**Similarity structure.** Ward clustering finds three natural pairs — {CayusePass, Paradise} (high-elevation Cascade sites, the tightest pair at distance 2.23), {Spokane, SourdoughGulch} (dry east-side, 4.14), and {Darrington, Quinault} (wet west-side, 7.58) — with BeaverPass joining the high-elevation group (k = 3). SourdoughGulch is the **only grassland** station (all others tree cover), i.e. the most unique by land cover.

**Hypothesis test.** The simple "twin → easy" story is **not supported** by the aggregate difficulty: 1-NN distance correlates *positively* with median LOSO R² (ρ = +0.73), i.e. the closest twins (CayusePass ↔ Paradise) are among the *hardest* stations on average. Under the best spatial generalizer (now the `Clustering_Backbone54_k2` winner) the pattern flips toward the hypothesis — mean-distance isolation correlates *negatively* with winner-config R² (ρ = −0.64), and the unique grassland station SourdoughGulch is the hardest (0.43) under both measures — but none of these correlations is significant at n = 7.

**Joint reading.** Having a static-feature twin does not rescue generalization when the twin pair is jointly *extreme* (CayusePass/Paradise: highest elevation, coldest, snow-dominated); per-station difficulty is better explained by how far a station sits from the training-regime core and by its dynamic regime (SourdoughGulch's unique land cover / low vegetation) than by pairwise similarity alone. These results are descriptive for a 7-station sample and should be treated as hypotheses, not conclusions.


## Full-Training Baseline — Intrinsic vs. Generalization Difficulty

LOSO measures how well a model *generalizes* to a station it never saw during training. But a station can be hard for two different reasons: it is **intrinsically hard** (hard to fit even when its own rows are in the training set) or **generalization-limited** (easy to fit when trained on, but degrades when held out). To separate the two, every configuration was also trained **without LOSO** — router and experts fit on the full trainval (all 7 stations), exactly the `derived_8.4-eval-1.1` protocol — and evaluated per station on the test set (`run_full_baseline.py`, same 56 configurations as LOSO). Pooled R² per configuration replicates eval-1.1 to within machine precision for the 47 pinned configurations, validating the baseline; the 9 new `Clustering_Backbone54_k2` configurations have no eval-1.1 row and their pooled R² serves as their temporal reference.


In [ ]:
df_full_pcs = pd.read_csv(EXP_DIR / "full_per_config_station.csv")
df_full_summary = pd.read_csv(EXP_DIR / "full_config_summary.csv")
df_full_station = pd.read_csv(EXP_DIR / "full_station_summary.csv")

valid = df_full_summary.dropna(subset=["eval11_test_r2"])
print(f"Validation vs eval-1.1: {len(valid)} configs compared, "
      f"max |full_pooled_r2 − eval11_test_r2| = {valid['r2_diff'].max():.6f}")

full_cols = ["station", "n_configs", "total_test_n", "median_r2", "mean_r2", "std_r2",
             "min_r2", "max_r2", "mean_rmse", "mean_bias", "n_negative_r2"]
print("\n### Station difficulty under FULL training (intrinsic; median over 56 configs)")
print(df_full_station[full_cols].to_markdown(index=False))


### Full-training vs LOSO difficulty

Merging the per-station medians: `median_r2_full` (trained on all stations) vs `median_r2_loso` (station held out). `gap = full − loso` is the LOSO cost — a large positive gap means the station is **generalization-limited** (fine when trained on, degrades when held out); a gap near zero means it is **intrinsically hard** (hard both ways); a negative gap means it performs *better* under LOSO than full training (anomaly worth flagging).


In [ ]:
cmp = df_full_station[["station", "median_r2", "mean_r2"]].merge(
    df_station[["station", "median_r2", "mean_r2"]],
    on="station", suffixes=("_full", "_loso"),
)
cmp["gap_median"] = cmp["median_r2_full"] - cmp["median_r2_loso"]
cmp["gap_mean"] = cmp["mean_r2_full"] - cmp["mean_r2_loso"]
cmp = cmp.sort_values("median_r2_loso", ascending=False)
cols = ["station", "median_r2_full", "median_r2_loso", "gap_median",
        "mean_r2_full", "mean_r2_loso", "gap_mean"]
print("### Per-station difficulty: full training vs LOSO (sorted by LOSO difficulty)")
print(cmp[cols].round(3).to_markdown(index=False))


In [ ]:
from scipy.stats import spearmanr
rho, p = spearmanr(cmp["median_r2_full"], cmp["median_r2_loso"])
print(f"Spearman(full median R², LOSO median R²) = {rho:+.3f} (p = {p:.3f}, n = {len(cmp)})")
print("High ρ ⇒ station difficulty is intrinsic; low ρ ⇒ LOSO difficulty is generalization-specific.")


### Figures

Three figures compare the two difficulty measures: (1) scatter of per-station median R² (full vs LOSO) with the identity line, (2) paired bars per station sorted by LOSO difficulty, and (3) per-station boxplot of the per-configuration LOSO cost (full-training R² − LOSO R²).


In [ ]:
from eval13.plots import plot_full_vs_loso_scatter, plot_full_vs_loso_bars, plot_loso_gap_boxplot

cmp_plot = cmp.rename(columns={"median_r2_full": "full_median_r2", "median_r2_loso": "loso_median_r2"})
for fig_path in [
    plot_full_vs_loso_scatter(cmp_plot, EXP_DIR),
    plot_full_vs_loso_bars(cmp_plot, EXP_DIR),
    plot_loso_gap_boxplot(df_full_pcs, df_pcs, EXP_DIR),
]:
    display(Image(filename=str(fig_path)))
print("Saved 3 full-baseline comparison figures.")


### Interpretation

**Replication.** All 47 pinned configurations reproduce eval-1.1's pooled test R² exactly (max |diff| = 0.000000), so the full-training baseline is a faithful replica of 1.1 — LOSO is a true addition, not a replacement. The 9 new `Clustering_Backbone54_k2` configurations get their temporal test R² from this baseline.

**Are LOSO-hard stations also hard under full training? Only weakly.** Spearman(full median R², LOSO median R²) = +0.286 (p = 0.535, n = 7): the station-difficulty rank order changes substantially between the two protocols, so LOSO difficulty is mostly *generalization-specific* rather than intrinsic.

**The two hardest LOSO stations split cleanly:**
- **CayusePass** is **generalization-limited**: easy when trained on (full median 0.752) yet the hardest under LOSO (0.350) — gap +0.40. Its snow-dominated high-elevation regime is fit well when its own rows are present, but the other stations cannot supply that regime when it is held out.
- **SourdoughGulch** is **intrinsically hard**: 0.477 full vs 0.431 LOSO — gap only +0.05. Even with its own rows in training it remains the joint-hardest station (with BeaverPass), consistent with its uniqueness (the only grassland station).

**BeaverPass anomaly.** BeaverPass is *harder* under full training (median 0.478) than under LOSO (0.682), gap −0.20 — the only station that does *better* when held out. It has the fewest test rows (626) and its regime overlaps few training rows, so including it in full training appears to hurt rather than help.

**Bottom line.** Station difficulty is not one-dimensional: LOSO difficulty measures *transfer*, full-training difficulty measures *fitability*. Only SourdoughGulch is hard on both axes (intrinsically hard *and* hard to generalize to); CayusePass's difficulty is almost entirely a transfer phenomenon.
